# 14. Three-Model Pivot Transfer Architecture (E10)

Executes Model A (Eng-Swa), Model B (Eng-Eke via A), and Model C (Swa-Eke via A).

In [4]:
%load_ext autoreload
%autoreload 2

import os, sys, gc, torch
sys.path.append(os.path.abspath('..'))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# ============================================================
# PATH & ENVIRONMENT BOOSTER — Guarantees project path setup
# ============================================================
import os, sys, site, urllib.request, zipfile, glob

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
for conda_site in glob.glob('/opt/conda/lib/python3.*/site-packages'):
    if conda_site not in sys.path:
        sys.path.insert(0, conda_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home     = os.path.expanduser('~')
proj_dir = os.path.join(home, 'Ekegusii-LLM-Translation-main')
tag_file = os.path.join(proj_dir, 'configs', 'models', 'v5_ready.tag')

# Download ONLY if tag_file is missing (prevents file modification during active runs)
if not os.path.isfile(tag_file):
    try:
        print('🔄 Syncing code from GitHub main branch...')
        zip_path = os.path.join(home, 'repo.zip')
        urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(home)
        os.remove(zip_path)
        print('✅ Code synced to latest version!')
    except Exception as exc:
        print(f'⚠️ Notice: {exc} (using local files)')
else:
    print('✅ Codebase up to date.')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


✅ Codebase up to date.
Working Directory : /home/jovyan/Ekegusii-LLM-Translation-main
Python Kernel     : /opt/conda/bin/python


In [11]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [12]:
import os, sys, importlib

print("🔧 Updating src/cli/train.py on server for E10 Model A, B, C...")

train_py_path = "src/cli/train.py"
if os.path.exists(train_py_path):
    with open(train_py_path, "r", encoding="utf-8") as f:
        content = f.read()
        
    if "E10_Model_A_English_Swahili" not in content:
        old_str = '"E9_Sequential_Transfer": SequentialTransferExperiment,'
        new_str = '"E9_Sequential_Transfer": SequentialTransferExperiment,\n    "E10_Model_A_English_Swahili": lambda: BilingualExperiment("eng_swa"),\n    "E10_Model_B_English_Ekegusii": lambda: BilingualExperiment("eng_eke"),\n    "E10_Model_C_Swahili_Ekegusii": lambda: BilingualExperiment("swa_eke"),'
        content = content.replace(old_str, new_str)
        
        with open(train_py_path, "w", encoding="utf-8") as f:
            f.write(content)
        print("  ✅ Patched src/cli/train.py successfully!")
    else:
        print("  ✅ src/cli/train.py already has E10 registered.")

# Reload module in active Python kernel memory
if 'src.cli.train' in sys.modules:
    importlib.reload(sys.modules['src.cli.train'])
    print("  ✅ Reloaded src.cli.train in kernel memory!")

print("\n🎉 SUCCESS! E10 Model A, B, C are registered and ready to train!")


🔧 Updating src/cli/train.py on server for E10 Model A, B, C...
  ✅ Patched src/cli/train.py successfully!
  ✅ Reloaded src.cli.train in kernel memory!

🎉 SUCCESS! E10 Model A, B, C are registered and ready to train!


In [13]:
from src.cli.train import run_train

print('🚀 Step 1: Training Model A (English ↔ Kiswahili Pivot Model)...')
run_train('E10_Model_A_English_Swahili', model_name='qwen')

print('🚀 Step 2: Training Model B (English ↔ Ekegusii via Model A)...')
run_train('E10_Model_B_English_Ekegusii', model_name='qwen')

print('🚀 Step 3: Training Model C (Kiswahili ↔ Ekegusii via Model A)...')
run_train('E10_Model_C_Swahili_Ekegusii', model_name='qwen')


🚀 Step 1: Training Model A (English ↔ Kiswahili Pivot Model)...


ValueError: mode must be one of ['eng_eke', 'swa_eke', 'combined'], got 'eng_swa'.

In [14]:
from src.cli.train import run_train

# Step 1: Train Model A
run_train("E10_Model_A_English_Swahili", model_name="qwen")

# Step 2: Train Model B
run_train("E10_Model_B_English_Ekegusii", model_name="qwen")

# Step 3: Train Model C
run_train("E10_Model_C_Swahili_Ekegusii", model_name="qwen")


ValueError: mode must be one of ['eng_eke', 'swa_eke', 'combined'], got 'eng_swa'.